# 🚗 Train Indian Vehicle Detection Model

This notebook fine-tunes **YOLOv8n** on the **Indian Driving Dataset (IDD)** with 41,962 images and 15 vehicle classes.

**Steps:**
1. Install dependencies
2. Download Indian Driving Dataset from Kaggle
3. Fine-tune YOLOv8n (~30 min on T4 GPU)
4. Download the trained model (`indian_yolov8n.pt`)

⚠️ **Make sure to select GPU runtime**: Runtime → Change runtime type → T4 GPU

## Step 1: Install Dependencies

In [ ]:
!pip install ultralytics kaggle -q

import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU")

## Step 2: Setup Kaggle & Download Dataset

Enter your Kaggle credentials below. You can find them at [kaggle.com/settings](https://kaggle.com/settings).

In [ ]:
import os

# ── Enter your Kaggle credentials here ──
os.environ['KAGGLE_USERNAME'] = 'satvikj2005'  # Your Kaggle username
os.environ['KAGGLE_KEY'] = ''  # Paste your Kaggle API key here

# Create kaggle.json
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
import json
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': os.environ['KAGGLE_USERNAME'], 'key': os.environ['KAGGLE_KEY']}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

print('✅ Kaggle credentials configured!')

In [ ]:
# Download the Indian Driving Dataset (22 GB - takes ~5 min on Colab)
!kaggle datasets download -d redzapdos123/indian-driving-dataset-detections-yolov11 -p /content/dataset --unzip
print('\n✅ Dataset downloaded!')

In [ ]:
# Explore the dataset structure
import os

dataset_root = '/content/dataset'
for root, dirs, files in os.walk(dataset_root):
    level = root.replace(dataset_root, '').count(os.sep)
    indent = ' ' * 2 * level
    file_count = len(files)
    if level < 3:
        print(f'{indent}{os.path.basename(root)}/ ({file_count} files)')

# Find data.yaml
import glob
yaml_files = glob.glob(os.path.join(dataset_root, '**', 'data.yaml'), recursive=True)
if yaml_files:
    print(f'\n📄 Found data.yaml at: {yaml_files[0]}')
    with open(yaml_files[0]) as f:
        print(f.read())
else:
    print('⚠️ data.yaml not found, will create one')

## Step 3: Configure & Train

Fine-tune YOLOv8n with the Indian vehicle dataset. Training should take ~20-30 minutes on a T4 GPU.

In [ ]:
import yaml
import glob
import os

dataset_root = '/content/dataset'

# Find the data.yaml
yaml_files = glob.glob(os.path.join(dataset_root, '**', 'data.yaml'), recursive=True)

if yaml_files:
    data_yaml_path = yaml_files[0]
    # Fix paths in data.yaml to point to the correct location
    with open(data_yaml_path) as f:
        data_config = yaml.safe_load(f)
    
    # Update paths
    yaml_dir = os.path.dirname(data_yaml_path)
    data_config['path'] = yaml_dir
    
    # Write updated config
    with open(data_yaml_path, 'w') as f:
        yaml.dump(data_config, f)
    
    print(f'📄 Using: {data_yaml_path}')
    print(f'📊 Classes: {data_config.get("names", "unknown")}')
    print(f'📁 Path: {data_config.get("path", "unknown")}')
else:
    # Create data.yaml if not found
    # Look for train/val directories
    train_dirs = glob.glob(os.path.join(dataset_root, '**', 'train', 'images'), recursive=True)
    val_dirs = glob.glob(os.path.join(dataset_root, '**', 'val', 'images'), recursive=True)
    
    print(f'Train dirs found: {train_dirs}')
    print(f'Val dirs found: {val_dirs}')
    
    if train_dirs:
        parent = os.path.dirname(os.path.dirname(train_dirs[0]))
        data_config = {
            'path': parent,
            'train': 'train/images',
            'val': 'val/images' if val_dirs else 'train/images',
            'nc': 15,
            'names': ['car', 'truck', 'bus', 'motorcycle', 'bicycle', 
                      'autorickshaw', 'animal', 'rider', 'traffic_sign',
                      'traffic_light', 'person', 'train', 'caravan', 
                      'vehicle_fallback', 'trailer']
        }
        data_yaml_path = os.path.join(parent, 'data.yaml')
        with open(data_yaml_path, 'w') as f:
            yaml.dump(data_config, f)
        print(f'Created data.yaml at: {data_yaml_path}')

In [ ]:
from ultralytics import YOLO

# Load pre-trained YOLOv8n
model = YOLO('yolov8n.pt')

# Fine-tune on Indian vehicle data
results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,        # Early stopping
    device=0,            # GPU
    workers=4,
    project='/content/runs',
    name='indian_vehicles',
    exist_ok=True,
    pretrained=True,
    verbose=True,
    plots=True,
)

print('\n✅ Training complete!')

## Step 4: Evaluate Results

In [ ]:
from IPython.display import Image, display
import os

results_dir = '/content/runs/indian_vehicles'

# Show training curves
if os.path.exists(os.path.join(results_dir, 'results.png')):
    display(Image(os.path.join(results_dir, 'results.png'), width=800))

# Show confusion matrix
if os.path.exists(os.path.join(results_dir, 'confusion_matrix.png')):
    display(Image(os.path.join(results_dir, 'confusion_matrix.png'), width=600))

# Show validation predictions
if os.path.exists(os.path.join(results_dir, 'val_batch0_pred.jpg')):
    display(Image(os.path.join(results_dir, 'val_batch0_pred.jpg'), width=800))

## Step 5: Download Trained Model

Download `indian_yolov8n.pt` and place it in your project's `backend/` folder.

In [ ]:
import shutil
from google.colab import files

# Copy best weights
best_weights = '/content/runs/indian_vehicles/weights/best.pt'
output_path = '/content/indian_yolov8n.pt'

if os.path.exists(best_weights):
    shutil.copy2(best_weights, output_path)
    
    # Get file size
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f'✅ Model ready: indian_yolov8n.pt ({size_mb:.1f} MB)')
    print()
    print('📥 Downloading...')
    print('After download, place the file in your project:')
    print('  parking-survey/backend/indian_yolov8n.pt')
    print()
    print('The pipeline will auto-detect and use it!')
    
    # Trigger download
    files.download(output_path)
else:
    # Try last.pt instead
    last_weights = '/content/runs/indian_vehicles/weights/last.pt'
    if os.path.exists(last_weights):
        shutil.copy2(last_weights, output_path)
        files.download(output_path)
    else:
        print('❌ No weights found. Check training output above for errors.')